
# Traffic Demand Prediction - Fully Regenerated Version

This notebook fixes:
- LightGBM dtype errors
- Object/string column issues
- Missing values
- Encoding pipeline

Models:
- LightGBM
- XGBoost
- CatBoost
- ExtraTrees
- HistGradientBoosting
- Ridge Stacking


In [1]:

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd

from sklearn.model_selection import KFold
from sklearn.metrics import r2_score
from sklearn.linear_model import Ridge
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.ensemble import HistGradientBoostingRegressor

from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
from catboost import CatBoostRegressor


In [2]:

train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')

TARGET='demand'
ID_COL='Index'

print(train.shape)
print(test.shape)


(77299, 11)
(41778, 10)


In [3]:

def build_features(df):

    df = df.copy()

    # timestamp
    parts = df['timestamp'].astype(str).str.split(':', expand=True)

    df['hour'] = parts[0].astype(int)
    df['minute'] = parts[1].astype(int)

    df['hour_sin'] = np.sin(2*np.pi*df['hour']/24)
    df['hour_cos'] = np.cos(2*np.pi*df['hour']/24)

    df['minute_sin'] = np.sin(2*np.pi*df['minute']/60)
    df['minute_cos'] = np.cos(2*np.pi*df['minute']/60)

    # geohash hierarchy
    for i in range(6):
        df[f'geo_{i}'] = (
            df['geohash']
            .astype(str)
            .str[i]
            .fillna('M')
        )

    df['geo_prefix2'] = df['geohash'].astype(str).str[:2]
    df['geo_prefix3'] = df['geohash'].astype(str).str[:3]
    df['geo_prefix4'] = df['geohash'].astype(str).str[:4]

    # interactions
    df['road_lane'] = (
        df['RoadType'].astype(str) +
        '_' +
        df['NumberofLanes'].astype(str)
    )

    df['weather_hour'] = (
        df['Weather'].astype(str) +
        '_' +
        df['hour'].astype(str)
    )

    return df

train = build_features(train)
test = build_features(test)


In [4]:
# =====================================================
# MISSING VALUE HANDLING
# =====================================================

for col in train.columns:

    if col == TARGET:
        continue

    # categorical/string columns
    if (
        pd.api.types.is_object_dtype(train[col])
        or pd.api.types.is_string_dtype(train[col])
        or pd.api.types.is_categorical_dtype(train[col])
    ):

        train[col] = train[col].fillna("Unknown")

        if col in test.columns:
            test[col] = test[col].fillna("Unknown")

    # numeric columns
    else:

        median_value = train[col].median()

        train[col] = train[col].fillna(median_value)

        if col in test.columns:
            test[col] = test[col].fillna(median_value)

print("Missing values handled")

Missing values handled


In [5]:

# Frequency Encoding

cat_cols = train.select_dtypes(include='object').columns.tolist()

for col in cat_cols:

    freq = train[col].value_counts()

    train[col+'_freq'] = train[col].map(freq)
    test[col+'_freq'] = test[col].map(freq).fillna(0)


In [6]:

# Target Encoding

kf = KFold(n_splits=5, shuffle=True, random_state=42)

for col in cat_cols:

    train[col+'_te'] = np.nan

    global_mean = train[TARGET].mean()

    for tr_idx,val_idx in kf.split(train):

        tr = train.iloc[tr_idx]
        val = train.iloc[val_idx]

        mapping = tr.groupby(col)[TARGET].mean()

        train.loc[val.index,col+'_te'] = val[col].map(mapping)

    train[col+'_te'] = train[col+'_te'].fillna(global_mean)

    mapping = train.groupby(col)[TARGET].mean()

    test[col+'_te'] = test[col].map(mapping).fillna(global_mean)


In [7]:

# Geohash Statistics

stats = train.groupby('geohash')[TARGET].agg(
    ['mean','median','std','count']
)

stats.columns = [
    'geo_mean',
    'geo_median',
    'geo_std',
    'geo_count'
]

train = train.merge(stats,on='geohash',how='left')
test = test.merge(stats,on='geohash',how='left')


In [8]:

# SAFE NUMERIC CONVERSION

y = train[TARGET].copy()

train_features = train.drop(columns=[TARGET])

full_df = pd.concat(
    [train_features,test],
    axis=0,
    ignore_index=True
)

object_cols = full_df.select_dtypes(include='object').columns

for col in object_cols:

    full_df[col] = (
        full_df[col]
        .astype(str)
        .fillna('Unknown')
        .astype('category')
        .cat.codes
        .astype('int32')
    )

train_encoded = full_df.iloc[:len(train)].copy()
test_encoded = full_df.iloc[len(train):].copy()

X = train_encoded.drop(columns=[ID_COL],errors='ignore')
X_test = test_encoded.drop(columns=[ID_COL],errors='ignore')

for col in X.columns:

    X[col] = pd.to_numeric(X[col], errors='coerce')
    X_test[col] = pd.to_numeric(X_test[col], errors='coerce')

X = X.fillna(0)
X_test = X_test.fillna(0)

print(X.select_dtypes(include='object').columns.tolist())

assert len(X.select_dtypes(include='object').columns) == 0


[]


In [9]:

N_FOLDS = 5

kf = KFold(
    n_splits=N_FOLDS,
    shuffle=True,
    random_state=42
)

models = ['lgb','xgb','cat','etr','hgb']

oof = {m:np.zeros(len(X)) for m in models}
pred = {m:np.zeros(len(X_test)) for m in models}


In [10]:

for fold,(tr_idx,val_idx) in enumerate(kf.split(X),1):

    print(f'Fold {fold}')

    X_tr = X.iloc[tr_idx]
    X_val = X.iloc[val_idx]

    y_tr = y.iloc[tr_idx]
    y_val = y.iloc[val_idx]

    lgb_model = LGBMRegressor(
        n_estimators=1500,
        learning_rate=0.02,
        num_leaves=255,
        subsample=0.85,
        colsample_bytree=0.85,
        random_state=42
    )

    lgb_model.fit(X_tr,y_tr)

    oof['lgb'][val_idx] = lgb_model.predict(X_val)
    pred['lgb'] += lgb_model.predict(X_test)/N_FOLDS

    xgb_model = XGBRegressor(
        n_estimators=1500,
        learning_rate=0.02,
        max_depth=10,
        subsample=0.85,
        colsample_bytree=0.85,
        tree_method='hist',
        random_state=42
    )

    xgb_model.fit(X_tr,y_tr)

    oof['xgb'][val_idx] = xgb_model.predict(X_val)
    pred['xgb'] += xgb_model.predict(X_test)/N_FOLDS

    cat_model = CatBoostRegressor(
        iterations=1500,
        depth=10,
        learning_rate=0.03,
        verbose=False
    )

    cat_model.fit(X_tr,y_tr)

    oof['cat'][val_idx] = cat_model.predict(X_val)
    pred['cat'] += cat_model.predict(X_test)/N_FOLDS

    etr = ExtraTreesRegressor(
        n_estimators=800,
        min_samples_leaf=2,
        n_jobs=-1,
        random_state=42
    )

    etr.fit(X_tr,y_tr)

    oof['etr'][val_idx] = etr.predict(X_val)
    pred['etr'] += etr.predict(X_test)/N_FOLDS

    hgb = HistGradientBoostingRegressor(
        learning_rate=0.03,
        max_depth=10,
        max_iter=1000,
        random_state=42
    )

    hgb.fit(X_tr,y_tr)

    oof['hgb'][val_idx] = hgb.predict(X_val)
    pred['hgb'] += hgb.predict(X_test)/N_FOLDS


Fold 1
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001521 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 3311
[LightGBM] [Info] Number of data points in the train set: 61839, number of used features: 54
[LightGBM] [Info] Start training from score 0.093784
Fold 2
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.002678 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 3312
[LightGBM] [Info] Number of data points in the train set: 61839, number of used features: 54
[LightGBM] [Info] Start training from score 0.093797
Fold 3
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.002687 seconds.
You can set `force_row_wise=true` to remove the overhead.
And i

In [11]:

for m in models:
    print(m, r2_score(y,oof[m]))


lgb 0.9565711023368535
xgb 0.9546129929553918
cat 0.9565615556485935
etr 0.9597121439279996
hgb 0.9509433719361158


In [12]:

stack_train = np.column_stack([
    oof['lgb'],
    oof['xgb'],
    oof['cat'],
    oof['etr'],
    oof['hgb']
])

stack_test = np.column_stack([
    pred['lgb'],
    pred['xgb'],
    pred['cat'],
    pred['etr'],
    pred['hgb']
])

meta = Ridge(alpha=1.0)
meta.fit(stack_train,y)

final_pred = meta.predict(stack_test)


In [13]:

submission = pd.DataFrame({
    'Index': test['Index'],
    'demand': final_pred
})

submission.to_csv('submission_final.csv',index=False)

print('submission_final.csv saved')


submission_final.csv saved
